# FIFA World Cup 2026 — Winner Prediction
## Notebook 05 — Streamlit Prep & Dashboard

The goal of this notebook is to prepare all data assets needed by the Streamlit dashboard and build the app/app.py file.

### Objectives
- Load and validate simulation results
- Compute Monte Carlo insights (most common final, biggest upsets)
- Export all dashboard-ready assets to data/processed/
- Build the full Streamlit dashboard in app/app.py

In [1]:
import pandas as pd
import numpy as np
import json
import os

pd.set_option('display.max_columns', None)

## Load Simulation Results

In [2]:
df_results = pd.read_csv("../data/processed/df_simulation_results.csv")
df_teams   = pd.read_csv("../data/processed/df_teams_2026_clean.csv")
df_matches = pd.read_csv("../data/processed/df_matches_2026_features.csv")

print(f"Simulation results: {df_results.shape}")
print()
print(df_results.head(10).to_string(index=False))

Simulation results: (48, 5)

       team  win_prob  final_prob  semi_final_prob  quarter_final_prob
  Argentina    0.3365      0.4321           0.4856              0.5311
      Spain    0.2908      0.4279           0.4823              0.5476
     France    0.1440      0.2501           0.5871              0.7363
    England    0.0802      0.1696           0.4203              0.6569
    Croatia    0.0276      0.0963           0.1837              0.3415
     Norway    0.0201      0.0812           0.1991              0.3602
     Turkey    0.0195      0.1059           0.1669              0.4438
Switzerland    0.0180      0.0751           0.2180              0.4563
   Portugal    0.0129      0.0587           0.1854              0.4501
   Paraguay    0.0097      0.0651           0.1293              0.4279


## Validate Results

In [3]:
# All 48 teams should be present
assert len(df_results) == 48, f"Expected 48 teams, got {len(df_results)}"

# Probabilities should sum to ~1
assert abs(df_results['win_prob'].sum() - 1.0) < 0.01, "Win probabilities don't sum to 1"

print("Validation passed.")
print(f"Total win probability: {df_results['win_prob'].sum():.4f}")
print(f"Top team: {df_results.iloc[0]['team']} ({df_results.iloc[0]['win_prob']:.2%})")

Validation passed.
Total win probability: 1.0000
Top team: Argentina (33.65%)


## Compute Monte Carlo Insights

In [4]:
# Top 5 contenders with all stage probabilities
top5 = df_results.head(5).copy()
top5['win_prob_pct']           = (top5['win_prob'] * 100).round(1)
top5['final_prob_pct']         = (top5['final_prob'] * 100).round(1)
top5['semi_final_prob_pct']    = (top5['semi_final_prob'] * 100).round(1)
top5['quarter_final_prob_pct'] = (top5['quarter_final_prob'] * 100).round(1)

print("Top 5 contenders:")
print(top5[['team', 'win_prob_pct', 'final_prob_pct', 'semi_final_prob_pct', 'quarter_final_prob_pct']].to_string(index=False))

Top 5 contenders:
     team  win_prob_pct  final_prob_pct  semi_final_prob_pct  quarter_final_prob_pct
Argentina          33.7            43.2                 48.6                    53.1
    Spain          29.1            42.8                 48.2                    54.8
   France          14.4            25.0                 58.7                    73.6
  England           8.0            17.0                 42.0                    65.7
  Croatia           2.8             9.6                 18.4                    34.2


In [5]:
# Teams with high quarter-final probability but low win probability
# (strong enough to go far but unlikely to win it all)
df_results['upset_score'] = df_results['quarter_final_prob'] / (df_results['win_prob'] + 0.001)

upsets = df_results[df_results['win_prob'] < 0.05].sort_values('upset_score', ascending=False).head(5)

print("Biggest potential upsets (high QF prob, low win prob):")
print(upsets[['team', 'win_prob', 'quarter_final_prob', 'upset_score']].to_string(index=False))

Biggest potential upsets (high QF prob, low win prob):
     team  win_prob  quarter_final_prob  upset_score
  Senegal    0.0002              0.1745   145.416667
Australia    0.0003              0.1875   144.230769
   Brazil    0.0003              0.1057    81.307692
   Canada    0.0024              0.2747    80.794118
  Germany    0.0030              0.3092    77.300000


In [6]:
# Teams with semi-final probability > 10% but win probability < 5%
dark_horses = df_results[
    (df_results['semi_final_prob'] > 0.10) &
    (df_results['win_prob'] < 0.05)
].sort_values('semi_final_prob', ascending=False)

print("Dark horses (semi_final_prob > 10%, win_prob < 5%):")
print(dark_horses[['team', 'win_prob', 'semi_final_prob']].to_string(index=False))

Dark horses (semi_final_prob > 10%, win_prob < 5%):
       team  win_prob  semi_final_prob
Switzerland    0.0180           0.2180
     Norway    0.0201           0.1991
   Portugal    0.0129           0.1854
    Croatia    0.0276           0.1837
     Turkey    0.0195           0.1669
   Paraguay    0.0097           0.1293
    Germany    0.0030           0.1129


## Export Dashboard Assets

In [7]:
os.makedirs("../data/processed", exist_ok=True)

# Full results with percentage columns
df_export = df_results.copy()
df_export['win_pct']           = (df_export['win_prob'] * 100).round(2)
df_export['final_pct']         = (df_export['final_prob'] * 100).round(2)
df_export['semi_final_pct']    = (df_export['semi_final_prob'] * 100).round(2)
df_export['quarter_final_pct'] = (df_export['quarter_final_prob'] * 100).round(2)

df_export.to_csv("../data/processed/df_dashboard_data.csv", index=False)

# Insights JSON
insights = {
    'top_contenders': top5[['team', 'win_prob_pct']].to_dict(orient='records'),
    'dark_horses':    dark_horses[['team', 'semi_final_prob']].head(3).to_dict(orient='records'),
    'biggest_upsets': upsets[['team', 'quarter_final_prob']].head(3).to_dict(orient='records'),
}

with open("../data/processed/dashboard_insights.json", "w") as f:
    json.dump(insights, f, indent=2)

print("Exported:")
print("  → data/processed/df_dashboard_data.csv")
print("  → data/processed/dashboard_insights.json")

Exported:
  → data/processed/df_dashboard_data.csv
  → data/processed/dashboard_insights.json


## Build app/app.py

In [8]:
app_code = '''import streamlit as st
import pandas as pd
import numpy as np
import json
import plotly.express as px
import plotly.graph_objects as go

# ── Page config ─────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="WC 2026 — ML Prediction",
    page_icon="⚽",
    layout="wide"
)

# ── WC2026 Color Palette ─────────────────────────────────────────────────────
BLACK     = "#0A0A0A"
GOLD      = "#C9A84C"
WHITE     = "#FFFFFF"
RED       = "#E61D25"
BLUE      = "#2A398D"
GREEN     = "#3CAC3B"
DARK_GRAY = "#474A4A"
LIGHT_GRAY = "#D1D4D1"

# ── Custom CSS ───────────────────────────────────────────────────────────────
st.markdown("""
<style>
    .stApp { background-color: #0A0A0A; color: #FFFFFF; }
    .main-title {
        font-size: 2.8rem;
        font-weight: 800;
        color: #C9A84C;
        text-align: center;
        margin-bottom: 0.2rem;
    }
    .subtitle {
        font-size: 1.1rem;
        color: #D1D4D1;
        text-align: center;
        margin-bottom: 2rem;
    }
    .metric-card {
        background-color: #1A1A1A;
        border: 1px solid #C9A84C;
        border-radius: 12px;
        padding: 1.2rem;
        text-align: center;
    }
    .metric-team {
        font-size: 1.1rem;
        font-weight: 700;
        color: #C9A84C;
    }
    .metric-prob {
        font-size: 2rem;
        font-weight: 800;
        color: #FFFFFF;
    }
    .metric-label {
        font-size: 0.8rem;
        color: #D1D4D1;
    }
    .section-title {
        font-size: 1.5rem;
        font-weight: 700;
        color: #C9A84C;
        border-bottom: 2px solid #C9A84C;
        padding-bottom: 0.3rem;
        margin-bottom: 1rem;
    }
    .insight-card {
        background-color: #1A1A1A;
        border-left: 4px solid #C9A84C;
        border-radius: 8px;
        padding: 1rem;
        margin-bottom: 0.8rem;
    }
</style>
""", unsafe_allow_html=True)

# ── Load Data ────────────────────────────────────────────────────────────────
@st.cache_data
def load_data():
    df = pd.read_csv("data/processed/df_dashboard_data.csv")
    with open("data/processed/dashboard_insights.json") as f:
        insights = json.load(f)
    return df, insights

df, insights = load_data()

# ── Hero Section ─────────────────────────────────────────────────────────────
st.markdown(\'<div class="main-title">⚽ FIFA World Cup 2026</div>\', unsafe_allow_html=True)
st.markdown(\'<div class="subtitle">ML Winner Prediction — XGBoost + Monte Carlo (10,000 simulations)</div>\', unsafe_allow_html=True)

# Top 3 metric cards
top3 = df.head(3)
cols = st.columns(3)
medals = ["🥇", "🥈", "🥉"]
for i, (col, (_, row)) in enumerate(zip(cols, top3.iterrows())):
    with col:
        st.markdown(f"""
        <div class="metric-card">
            <div class="metric-team">{medals[i]} {row["team"]}</div>
            <div class="metric-prob">{row["win_pct"]}%</div>
            <div class="metric-label">Win Probability</div>
        </div>
        """, unsafe_allow_html=True)

st.markdown("<br>", unsafe_allow_html=True)

# ── Win Probability Chart ─────────────────────────────────────────────────────
st.markdown(\'<div class="section-title">Win Probability — Top 15</div>\', unsafe_allow_html=True)

top15 = df.head(15).sort_values("win_pct", ascending=True)

fig_bar = px.bar(
    top15,
    x="win_pct",
    y="team",
    orientation="h",
    text="win_pct",
    color="win_pct",
    color_continuous_scale=[[0, DARK_GRAY], [0.5, BLUE], [1, GOLD]],
)
fig_bar.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside",
)
fig_bar.update_layout(
    plot_bgcolor=BLACK,
    paper_bgcolor=BLACK,
    font_color=WHITE,
    xaxis_title="Win Probability (%)",
    yaxis_title="",
    coloraxis_showscale=False,
    height=500,
    margin=dict(l=20, r=60, t=20, b=20),
)
fig_bar.update_xaxes(gridcolor=DARK_GRAY)
fig_bar.update_yaxes(gridcolor=DARK_GRAY)

st.plotly_chart(fig_bar, use_container_width=True)

# ── Stage-by-Stage Table ──────────────────────────────────────────────────────
st.markdown(\'<div class="section-title">Stage-by-Stage Probabilities — All 48 Teams</div>\', unsafe_allow_html=True)

df_table = df[[
    "team", "win_pct", "final_pct", "semi_final_pct", "quarter_final_pct"
]].copy()
df_table.columns = ["Team", "Win %", "Final %", "Semi-Final %", "Quarter-Final %"]

st.dataframe(
    df_table,
    use_container_width=True,
    hide_index=True,
    column_config={
        "Win %":           st.column_config.ProgressColumn("Win %", min_value=0, max_value=100, format="%.1f%%"),
        "Final %":         st.column_config.ProgressColumn("Final %", min_value=0, max_value=100, format="%.1f%%"),
        "Semi-Final %":    st.column_config.ProgressColumn("Semi-Final %", min_value=0, max_value=100, format="%.1f%%"),
        "Quarter-Final %": st.column_config.ProgressColumn("Quarter-Final %", min_value=0, max_value=100, format="%.1f%%"),
    }
)

# ── Team Explorer ─────────────────────────────────────────────────────────────
st.markdown(\'<div class="section-title">Team Explorer</div>\', unsafe_allow_html=True)

selected_team = st.selectbox("Select a team", df["team"].tolist())
team_row = df[df["team"] == selected_team].iloc[0]

col1, col2, col3, col4 = st.columns(4)
with col1:
    st.metric("Win Tournament", f"{team_row[\'win_pct\']}%")
with col2:
    st.metric("Reach Final", f"{team_row[\'final_pct\']}%")
with col3:
    st.metric("Reach Semi-Final", f"{team_row[\'semi_final_pct\']}%")
with col4:
    st.metric("Reach Quarter-Final", f"{team_row[\'quarter_final_pct\']}%")

# Radar chart for selected team
categories = ["Win", "Final", "Semi-Final", "Quarter-Final"]
values = [
    team_row["win_pct"],
    team_row["final_pct"],
    team_row["semi_final_pct"],
    team_row["quarter_final_pct"],
]

fig_radar = go.Figure()
fig_radar.add_trace(go.Scatterpolar(
    r=values + [values[0]],
    theta=categories + [categories[0]],
    fill="toself",
    fillcolor="rgba(201, 168, 76, 0.2)",
    line=dict(color=GOLD, width=2),
    name=selected_team,
))
fig_radar.update_layout(
    polar=dict(
        bgcolor=BLACK,
        radialaxis=dict(visible=True, gridcolor=DARK_GRAY, color=LIGHT_GRAY),
        angularaxis=dict(gridcolor=DARK_GRAY, color=WHITE),
    ),
    paper_bgcolor=BLACK,
    font_color=WHITE,
    showlegend=False,
    height=400,
    margin=dict(l=40, r=40, t=40, b=40),
)
st.plotly_chart(fig_radar, use_container_width=True)

# ── Monte Carlo Insights ──────────────────────────────────────────────────────
st.markdown(\'<div class="section-title">Monte Carlo Insights</div>\', unsafe_allow_html=True)

col_a, col_b = st.columns(2)

with col_a:
    st.markdown("**🌟 Dark Horses**")
    for t in insights["dark_horses"]:
        prob = round(t["semi_final_prob"] * 100, 1)
        st.markdown(f"""
        <div class="insight-card">
            <strong style="color:{GOLD}">{t["team"]}</strong><br>
            <span style="color:{LIGHT_GRAY}">{prob}% chance of reaching Semi-Finals</span>
        </div>
        """, unsafe_allow_html=True)

with col_b:
    st.markdown("**⚡ Biggest Upsets**")
    for t in insights["biggest_upsets"]:
        prob = round(t["quarter_final_prob"] * 100, 1)
        st.markdown(f"""
        <div class="insight-card">
            <strong style="color:{RED}">{t["team"]}</strong><br>
            <span style="color:{LIGHT_GRAY}">{prob}% chance of reaching Quarter-Finals</span>
        </div>
        """, unsafe_allow_html=True)

# ── Footer ────────────────────────────────────────────────────────────────────
st.markdown("<br><br>", unsafe_allow_html=True)
st.markdown(
    \'<div style="text-align:center; color:#474A4A; font-size:0.8rem;">\' +
    "Built with XGBoost + Monte Carlo simulation · 10,000 tournament simulations · " +
    "Training data: 418 historical WC matches between 2026 qualified nations" +
    "</div>",
    unsafe_allow_html=True
)
'''

os.makedirs("../app", exist_ok=True)

with open("../app/app.py", "w") as f:
    f.write(app_code)

print("app/app.py written successfully.")

app/app.py written successfully.


## Install Plotly and Run Dashboard

In [9]:
import subprocess
result = subprocess.run(["uv", "add", "plotly"], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)


Using CPython 3.13.12
Creating virtual environment at: /Users/gianmarcoassandria/Desktop/Proyectos_DS/wc2026-predictor/.venv
Resolved 140 packages in 457ms
 Downloaded plotly
Prepared 1 package in 992ms
Installed 135 packages in 793ms
 + altair==6.2.1
 + anyio==4.13.0
 + appnope==0.1.4
 + argon2-cffi==25.1.0
 + argon2-cffi-bindings==25.1.0
 + arrow==1.4.0
 + asttokens==3.0.1
 + async-lru==2.3.0
 + attrs==26.1.0
 + babel==2.18.0
 + beautifulsoup4==4.15.0
 + bleach==6.4.0
 + blinker==1.9.0
 + cachetools==7.1.4
 + certifi==2026.5.20
 + cffi==2.0.0
 + charset-normalizer==3.4.7
 + click==8.4.1
 + comm==0.2.3
 + contourpy==1.3.3
 + cycler==0.12.1
 + debugpy==1.8.21
 + decorator==5.3.1
 + defusedxml==0.7.1
 + executing==2.2.1
 + fastjsonschema==2.21.2
 + fonttools==4.63.0
 + fqdn==1.5.1
 + gitdb==4.0.12
 + gitpython==3.1.50
 + h11==0.16.0
 + httpcore==1.0.9
 + httptools==0.8.0
 + httpx==0.28.1
 + idna==3.18
 + ipykernel==7.2.0
 + ipython==9.14.1
 + ipython-pygments-lexers==1.1.1
 + ipywidget

## Notebook 05 Complete

### Files exported
- `data/processed/df_dashboard_data.csv` — full results with percentage columns
- `data/processed/dashboard_insights.json` — dark horses and upset insights
- `app/app.py` — full Streamlit dashboard

### To run the dashboard
```bash
cd app
streamlit run app.py
```